# Fake News Detection 

In [1]:
import re
import html
import logging
from pathlib import Path
from urllib.parse import urlparse

import joblib
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


## Configuration

In [2]:
RANDOM_STATE = 42
TEST_SIZE = 0.20
MODEL_PATH = Path("fake_news_model.joblib")
REQUEST_TIMEOUT = 15

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
)
logger = logging.getLogger(__name__)


## Text Cleaning 

In [3]:
def clean_text(text):
    """Clean article text without removing useful word information."""
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")

    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


## Load and Validate Data

In [4]:
def load_dataset():
    true_path = Path("True.csv")
    fake_path = Path("Fake.csv")

    if not true_path.exists() or not fake_path.exists():
        raise FileNotFoundError(
            "True.csv and Fake.csv must be in the current directory."
        )

    true_df = pd.read_csv(true_path)
    fake_df = pd.read_csv(fake_path)

    if "text" not in true_df.columns or "text" not in fake_df.columns:
        raise ValueError("Both CSV files must contain a 'text' column.")

    true_df["label"] = 1
    fake_df["label"] = 0

    data = pd.concat([true_df, fake_df], ignore_index=True)
    data = data[["text", "label"]].copy()
    data["text"] = data["text"].fillna("").astype(str)
    data["text"] = data["text"].map(clean_text)

    # Remove empty documents.
    data = data[data["text"].str.len() > 0]
    
    # Remove exact duplicate texts to reduce memorization.
    data = data.drop_duplicates(subset="text")

    # Basic dataset validation.
    if data["label"].nunique() != 2:
        raise ValueError("The dataset must contain both classes.")

    if data["label"].value_counts().min() < 5:
        raise ValueError("Each class needs at least five examples.")
    data = data.sample(
        frac=1,
        random_state=RANDOM_STATE,
    ).reset_index(drop=True)

    print("\nDataset summary")
    print("----------------")
    print(f"Total articles: {len(data)}")
    print(f"Real articles:  {(data['label'] == 1).sum()}")
    print(f"Fake articles:  {(data['label'] == 0).sum()}")

    return data

## Model and Hyperparameter Search

In [5]:
def build_search():
    pipeline = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                sublinear_tf=True,
                strip_accents="unicode",
                max_df=0.95,
                min_df=2,
            ),
        ),
        ("classifier", LinearSVC()),
    ])
     # Compare algorithms and tune text features together.
    param_grid = [
        {
            "tfidf__analyzer": ["word"],
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [50000, 100000],
            "classifier": [
                LinearSVC(class_weight="balanced"),
            ],
            "classifier__C": [0.5, 1.0, 2.0],
        },
        {
            "tfidf__analyzer": ["word"],
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [50000],
            "classifier": [
                LogisticRegression(
                    max_iter=2000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ],
            "classifier__C": [0.5, 1.0, 2.0],
        },
        {
            "tfidf__analyzer": ["word"],
            "tfidf__ngram_range": [(1, 1), (1, 2)],
            "tfidf__max_features": [50000],
            "classifier": [MultinomialNB()],
            "classifier__alpha": [0.1, 0.5, 1.0],
        },
    ]
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1,
        return_train_score=False,
    )
    return search, cv


## Train, Validate and Evaluate

In [6]:
def train_model():
    data = load_dataset()

    X = data["text"]
    y = data["label"]

    # Reserve test data until model selection is complete.
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    search, cv = build_search()
    print("\nSearching for the best model...")
    search.fit(X_train, y_train)

    model = search.best_estimator_

    print("\nBest parameters:")
    print(search.best_params_)

    print(
        f"\nBest cross-validation macro F1: "
        f"{search.best_score_:.4f}"
    )
    # Evaluate once on untouched holdout data.
    y_pred = model.predict(X_test)

    print("\nHeld-out test results")
    print("---------------------")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(
        f"Precision (macro): "
        f"{precision_score(y_test, y_pred, average='macro', zero_division=0):.4f}"
    )
    print(
        f"Recall (macro): "
        f"{recall_score(y_test, y_pred, average='macro', zero_division=0):.4f}"
    )
    print(
        f"Macro F1:  "
        f"{f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}"
    )
    print("\nClassification report:")
    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1],
            target_names=["Fake", "Real"],
            zero_division=0,
        )
    )
    # Plot confusion matrix.
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Predicted Fake", "Predicted Real"],
        yticklabels=["Actual Fake", "Actual Real"],
    )
    plt.title("Fake News Detector - Test Confusion Matrix")
    plt.tight_layout()
    plt.show()
    # Show model comparison using cross-validation on training data.
    print("\nCross-validation comparison on training data:")
    print(f"Best tuned pipeline macro F1: {search.best_score_:.4f}")

    cv_results = pd.DataFrame(search.cv_results_)
    top_results = cv_results.sort_values(
        "rank_test_score"
    )[
        ["mean_test_score", "std_test_score", "params"]
    ].head(5)

    print(top_results.to_string(index=False))

    # Save the fitted pipeline, including its TF-IDF vectorizer.
    joblib.dump(model, MODEL_PATH)
    print(f"\nSaved model to: {MODEL_PATH.resolve()}")

    return model

## Load and Trained model

In [7]:
def get_model():
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            "Model not found. Run training first."
        )

    return joblib.load(MODEL_PATH)

## Predict News Text

In [12]:
def predict_news(news, model=None):
    if model is None:
        model = get_model()

    cleaned = clean_text(news)

    if len(cleaned.split()) < 5:
        return {
            "prediction": "Insufficient text",
            "warning": "Provide a longer article for analysis.",
        }

    prediction = int(model.predict([cleaned])[0])

    if prediction == 1:
        label = "Real-like (model prediction)"
    else:
        label = "Fake-like (model prediction)"
    return {
        "prediction": label,
        "warning": (
            "This is a text-classifier result, not proof of truth "
            "or falsity. Verify the claims independently."
        ),
    }

## Fetch live news from Google News RSS

In [13]:
def fetch_live_news(limit=10, query=None):
    if query:
        from urllib.parse import quote_plus
        url = (
            "https://news.google.com/rss/search?q="
            + quote_plus(query)
            + "&hl=en-IN&gl=IN&ceid=IN:en"
        )
    else:
        url = (
            "https://news.google.com/rss"
            "?hl=en-IN&gl=IN&ceid=IN:en"
        )

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; NewsResearchBot/1.0)"
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=REQUEST_TIMEOUT,
        )
        response.raise_for_status()

        soup = BeautifulSoup(response.content, "xml")
        items = []

        for item in soup.find_all("item")[:limit]:
            title_tag = item.find("title")
            link_tag = item.find("link")
            pub_date_tag = item.find("pubDate")
            source_tag = item.find("source")
            title = (
                title_tag.get_text(" ", strip=True)
                if title_tag else ""
            )
            link = (
                link_tag.get_text(strip=True)
                if link_tag else ""
            )

            if not title:
                continue
            items.append({
                "title": title,
                "link": link,
                "published": (
                    pub_date_tag.get_text(strip=True)
                    if pub_date_tag else "Unknown"
                ),
                "source": (
                    source_tag.get_text(strip=True)
                    if source_tag else "Unknown"
                ),
            })
            return items
    except requests.RequestException as exc:
        logger.error("Could not fetch RSS feed: %s", exc)
        return []
    except Exception as exc:
        logger.error("Could not parse RSS feed: %s", exc)
        return []


## Analyze News Headlines

In [14]:
def analyze_live_news(limit=10, query=None):
    model = get_model()
    articles = fetch_live_news(limit=limit, query=query)

    if not articles:
        print("No headlines were retrieved. Check your connection.")
        return

    print("\nLive headline analysis")
    print("======================")

    for index, article in enumerate(articles, start=1):
        result = predict_news(article["title"], model)

        print(f"\n{index}. {article['title']}")
        print(f"Source: {article['source']}")
        print(f"Published: {article['published']}")
        print(f"Model output: {result['prediction']}")
        print(f"Note: {result['warning']}")
        print(f"Link: {article['link']}")

## Interactive Menu

In [15]:
def main():
    print("FAKE NEWS DETECTION SYSTEM")
    print("==========================")
    print("1. Train / retrain model")
    print("2. Check a news article")
    print("3. Analyze latest headlines")
    print("4. Search live headlines by keyword")
    print("5. Exit")

    while True:
        choice = input("\nChoose an option (1-5): ").strip()

        if choice == "1":
            try:
                train_model()
            except Exception as exc:
                logger.exception("Training failed: %s", exc)
        elif choice == "2":
            try:
                text = input(
                    "Paste the full article or headline: "
                ).strip()

                result = predict_news(text)
                print("\nResult:", result["prediction"])
                print("Note:", result["warning"])

            except Exception as exc:
                logger.error("Prediction failed: %s", exc)
        elif choice == "3":
            analyze_live_news(limit=10)

        elif choice == "4":
            query = input(
                "Enter a topic or keywords to search: "
            ).strip()

            if query:
                analyze_live_news(limit=10, query=query)
            else:
                print("Please enter a search term.")

        elif choice == "5":
            print("Goodbye!")
            break

        else:
            print("Invalid choice. Enter a number from 1 to 5.")
if __name__ == "__main__":
    main()

FAKE NEWS DETECTION SYSTEM
1. Train / retrain model
2. Check a news article
3. Analyze latest headlines
4. Search live headlines by keyword
5. Exit

Live headline analysis

1. Cyclone Arnab crosses coast south of Yanam; landfall nearing completion, says IMD | LIVE - The Hindu
Source: The Hindu
Published: Wed, 28 Oct 2026 04:32:30 GMT
Model output: Fake-like (model prediction)
Note: This is a text-classifier result, not proof of truth or falsity. Verify the claims independently.
Link: https://news.google.com/rss/articles/CBMi8AFBVV95cUxPYndjRXNBX1gwWFAxNkhMX0JZVm9fNnFHQjB0ZVBpVEdacjM5dGVLNEZ5SVVkUnNVT0FDV2xIRnJ0UnRTS2ZyZ3p1R0dWM1V1WnFta0FTdFA3ZmtlZ1UyTUxFX2gxOFJTWlhHbFB6RS1MczJ2dU1hR0VqTFhMS3ljQ29xNVR0QnB6aDk5TXZFVG9QMjhwY3JiT0FmVUhmcE9FVllEMDFRU3VtRERYLTZFQ3B3VHVQWjI3ZHBwUGt4YXNoWV9xWVlSbEV1UTlValpMcWt6MTF6SVA3Rzlxal9xYUVUMTdSaFZLNlo5cGkzTWXSAfcBQVVfeXFMTlQxWENwZFdka2ViWDlDcGRjRFJXamlMRUZuWWFJMFhqUC1JdUJQQlM5N2k2bDlfbWkxbEcxdm52TWo0ZmRxR0NzbzJJdVFEcU9XUlNMeEJ5WGQ0OFh6cTJjZWhMWmt6ckhYRW